In [30]:
import urllib.request
import xml.etree.ElementTree as ET
import datetime
import neopixel
from neopixel import *
import board
import time



In [ ]:
class Color():
    red: int
    green: int
    blue: int
    def __init__(self, red: int, green: int, blue: int):
        self.red = int(red)
        self.green = int(green)
        self.blue = int(blue)

    def __iter__(self):
        return iter((self.red, self.green, self.blue))

    def __repr__(self):
        return f"Color({self.red}, {self.green}, {self.blue})"

In [23]:
# LED strip configuration:
LED_COUNT      = 248     # Number of LED pixels.
LED_PIN        = 18      # GPIO pin connected to the pixels (18 uses PWM!).
#LED_PIN        = 10      # GPIO pin connected to the pixels (10 uses SPI /dev/spidev0.0).
LED_FREQ_HZ    = 800000  # LED signal frequency in hertz (usually 800khz)
LED_DMA        = 10      # DMA channel to use for generating signal (try 5)
LED_BRIGHTNESS = 10      # Set to 0 for darkest and 255 for brightest
LED_INVERT     = False   # True to invert the signal (when using NPN transistor level shift)
LED_CHANNEL    = 0       # set to '1' for GPIOs 13, 19, 41, 45 or 53
#LED_STRIP      = ws.WS2811_STRIP_GRB   # Strip type and colour ordering

COLOR_VFR		= 	Color(255,0,0)		# Green
COLOR_VFR_FADE		= Color(125,0,0)		# Green Fade for wind
COLOR_MVFR		= 	Color(0,0,255)		# Blue
COLOR_MVFR_FADE		= Color(0,0,125)		# Blue Fade for wind
COLOR_IFR		= 	Color(0,255,0)		# Red
COLOR_IFR_FADE		= Color(0,125,0)		# Red Fade for wind
COLOR_LIFR		= 	Color(0,125,125)		# Magenta
COLOR_LIFR_FADE		= Color(0,75,75)		# Magenta Fade for wind
COLOR_UNK		= 	Color(255,255,255)		# Green
COLOR_UNK_FADE		= Color(255,255,255)		# Green Fade for wind
COLOR_CLEAR		= 	Color(0,0,0)		# Clear
COLOR_LIGHTNING		= Color(255,255,255)		# White

# ----- Blink/Fade functionality for Wind and Lightning -----
# Do you want the METARMap to be static to just show flight conditions, or do you also want blinking/fading based on current wind conditions
ACTIVATE_WINDCONDITION_ANIMATION = True	# Set this to False for Static or True for animated wind conditions
#Do you want the Map to Flash white for lightning in the area
ACTIVATE_LIGHTNING_ANIMATION = True		# Set this to False for Static or True for animated Lightning
# Fade instead of blink
FADE_INSTEAD_OF_BLINK	= False			# Set to False if you want blinking
# Blinking Windspeed Threshold
WIND_BLINK_THRESHOLD	= 10			# Knots of windspeed
ALWAYS_BLINK_FOR_GUSTS	= False			# Always animate for Gusts (regardless of speeds)
# Blinking Speed in seconds
BLINK_SPEED		= 1.0			# Float in seconds, e.g. 0.5 for half a second
# Total blinking time in seconds.
# For example set this to 300 to keep blinking for 5 minutes if you plan to run the script every 5 minutes to fetch the updated weather
BLINK_TOTALTIME_SECONDS	= 600

In [24]:
with open("airports") as f:
    airports = f.readlines()
airports = [x.strip() for x in airports]

url = "https://aviationweather.gov/api/data/metar?format=xml&hoursBeforeNow=5&mostRecentForEachStation=true&ids="

airport_list = [airportcode for airportcode in airports if airportcode != "NULL"]
url = url + ",".join(airport_list)

print (url)

https://aviationweather.gov/api/data/metar?format=xml&hoursBeforeNow=5&mostRecentForEachStation=true&ids=KIAH,KVCT,KCRP,KHRL,KLRD,KSAT,KAUS,KACT,KDFW,KOKC,KICT,KSLN,KGCK,KWWR,KCSM,KSPS,KABI,KSJT,KDLF,KFST,KMAF,KLBB,KCDS,KAMA,KCAO,KCVS,KROW,KELP,KDMN,KABQ,KALS,KCOS,KDEN,KEGE,KGJT,KDRO,KGUP,KSJN,KSAD,KTUS,KPHX,KGCN,KSGU,KLAS,KHII,KNYL,KSAN,KLAX,KSMX,KBFL,KNLC,KMCE,KSFO,KSMF,KRNO,KBIH,KTPH,KTMT,KELY,KDTA,KVEL,KSLC,KENV,KEKO,KWMC,KLKV,KLMT,KRDD,KACV,KOTH,KEUG,KRDM,KBNO,KBOI,KIDA,KSUN,KSMN,KMYL,KBKE,KPDX,KSEA,KYKM,KPSC,KMWH,KOMK,KGEG,KPUW,KMSO,KGPI,KCTB,KGTF,KBTM,KBZN,KPNA,KRKS,KRWL,KCYS,KCPR,KGCC,KSHR,KRIW,KCOD,KBIL,KLWT,KHVR,KGGW,KMLS,KGDV,KXWA,KDIK,KRAP,KAIA,KGLD,KLBF,KEAR,KOMA,KSUX,KFSD,KHON,KVTN,KPHP,KMBG,KBIS,KMOT,KDVL,KJMS,KABR,KFAR,KGFK,KINL,KBJI,KAXN,KBRD,KMSP,KDLH,KCKC,KIWD,KCMX,KSAW,KPLN,KAPN,KTVC,KGRB,KVOK,KEAU,KRST,KFOD,KDSM,KCID,KMLI,KPIA,KSTL,KDEC,KUIN,KIRK,KMCI,KCOU,KSGF,KXNA,KTUL,KMLC,KTXK,KSHV,KAEX,KARA,KMSY,KMLU,KGLH,KJAN,KMOB,KVPS,KMGM,KMEI,KBHM,KCBM,KMEM,KLIT,KARG,KPAH,

In [25]:
content = urllib.request.urlopen(url).read()
print (content)

b'<?xml version="1.0" encoding="UTF-8"?><response xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:noNamespaceSchemaLocation="https://aviationweather.gov/api/schema/metars2_0.xsd" version="2.0"><request_index>1762810547</request_index><data_source name="metars"/><request type="retrieve"/><errors/><warnings/><time_taken_ms>0</time_taken_ms><data num_results="244"><METAR><raw_text>SPECI KCKB 102133Z 23009KT 3/4SM -SN BR BKN006 M02/M03 A2992 RMK AO2 P0002 T10171028</raw_text><station_id>KCKB</station_id><observation_time>2025-11-10T21:33:00.000Z</observation_time><latitude>39.3022</latitude><longitude>-80.2239</longitude><temp_c>-1.7</temp_c><dewpoint_c>-2.8</dewpoint_c><wind_dir_degrees>230</wind_dir_degrees><wind_speed_kt>9</wind_speed_kt><visibility_statute_mi>0.75</visibility_statute_mi><altim_in_hg>29.92</altim_in_hg><quality_control_flags><auto_station>TRUE</auto_station></quality_control_flags><wx_string>-SN BR</wx_string><sky_c

In [31]:
root = ET.fromstring(content)
missingCondList=[]
conditionDict = { "NULL": {"flightCategory" : "", "windDir": "", "windSpeed" : 0, "windGustSpeed" :  0, "windGust" : False, "lightning": False, "tempC" : 0, "dewpointC" : 0, "vis" : 0, "altimHg" : 0, "obs" : "", "skyConditions" : {}, "obsTime" : datetime.datetime.now() } }
conditionDict.pop("NULL")

for metar in root.iter('METAR'):
	stationId = metar.find('station_id').text
	if metar.find('flight_category') is None:
		print ("Skipping " + stationId + " no flight category")
		missingCondList.append(stationId)
		continue

	flightCategory = metar.find('flight_category').text
	windDir = ""
	windSpeed = 0
	windGustSpeed = 0
	windGust = False
	lightning = False
	tempC = 0
	dewpointC = 0
	vis = 0
	altimHg = 0.0
	obs = ""
	skyConditions = []

	if metar.find('wind_gust_kt') is not None:
		windGustSpeed = int(metar.find('wind_gust_kt').text)
		windGust = (True if (ALWAYS_BLINK_FOR_GUSTS or windGustSpeed > WIND_BLINK_THRESHOLD) else False)
	if metar.find('wind_speed_kt') is not None:
		windSpeed = int(metar.find('wind_speed_kt').text)
	if metar.find('wind_dir_degrees') is not None:
		windDir = metar.find('wind_dir_degrees').text
	if metar.find('temp_c') is not None:
		tempC = int(round(float(metar.find('temp_c').text)))
	if metar.find('dewpoint_c') is not None:
		dewpointC = int(round(float(metar.find('dewpoint_c').text)))
	if metar.find('visibility_statute_mi') is not None:
		vis = int(round(float(metar.find('visibility_statute_mi').text.replace('+',''))))
	if metar.find('altim_in_hg') is not None:
		altimHg = float(round(float(metar.find('altim_in_hg').text), 2))
	if metar.find('wx_string') is not None:
		obs = metar.find('wx_string').text
	if metar.find('observation_time') is not None:
		obsTimeStr = metar.find('observation_time').text.replace("Z", "+00:00")
		obsTime = datetime.datetime.strptime(obsTimeStr.split('+')[0], "%Y-%m-%dT%H:%M:%S.%f")
	for skyIter in metar.iter("sky_condition"):
		skyCond = { "cover" : skyIter.get("sky_cover"), "cloudBaseFt": int(skyIter.get("cloud_base_ft_agl", default=0)) }
		skyConditions.append(skyCond)
	if metar.find('raw_text') is not None:
		rawText = metar.find('raw_text').text
		lightning = False if rawText.find('LTG') == -1 else True

	conditionDict[stationId] = { "flightCategory" : flightCategory, "windDir": windDir, "windSpeed" : windSpeed, "windGustSpeed": windGustSpeed, "windGust": windGust, "vis": vis, "obs" : obs, "tempC" : tempC, "dewpointC" : dewpointC, "altimHg" : altimHg, "lightning": lightning, "skyConditions" : skyConditions, "obsTime": obsTime }

print("Missing conditions for stations:", missingCondList)

Missing conditions for stations: []


In [34]:
looplimit = int(round(BLINK_TOTALTIME_SECONDS / BLINK_SPEED)) if (ACTIVATE_WINDCONDITION_ANIMATION or ACTIVATE_LIGHTNING_ANIMATION) else 1
windCycle = False
while looplimit > 0:
	print(f"Loop {looplimit} of {int(round(BLINK_TOTALTIME_SECONDS / BLINK_SPEED))}")
	i = 0

	# Set light color and status for all entries in airports list
	for airport in airports:
		color = COLOR_CLEAR
		conditions = conditionDict.get(airport, None)
		windy = False
		lightningConditions = False
		fltCat = conditions["flightCategory"] if conditions is not None else "None"
		if conditions != None:
			windy = True if (ACTIVATE_WINDCONDITION_ANIMATION and windCycle == True and (conditions["windSpeed"] > WIND_BLINK_THRESHOLD or conditions["windGust"] == True)) else False
			lightningConditions = True if (ACTIVATE_LIGHTNING_ANIMATION and windCycle == False and conditions["lightning"] == True) else False

			if conditions["flightCategory"] == "VFR":
				color = COLOR_VFR if not (windy or lightningConditions) else COLOR_LIGHTNING if lightningConditions else (COLOR_VFR_FADE if FADE_INSTEAD_OF_BLINK else COLOR_CLEAR) if windy else COLOR_CLEAR
				colorName = "Green"
			elif conditions["flightCategory"] == "MVFR":
				color = COLOR_MVFR if not (windy or lightningConditions) else COLOR_LIGHTNING if lightningConditions else (COLOR_MVFR_FADE if FADE_INSTEAD_OF_BLINK else COLOR_CLEAR) if windy else COLOR_CLEAR
				colorName = "Blue"
			elif conditions["flightCategory"] == "IFR":
				color = COLOR_IFR if not (windy or lightningConditions) else COLOR_LIGHTNING if lightningConditions else (COLOR_IFR_FADE if FADE_INSTEAD_OF_BLINK else COLOR_CLEAR) if windy else COLOR_CLEAR
				colorName = "Red"
			elif conditions["flightCategory"] == "LIFR":
				color = COLOR_LIFR if not (windy or lightningConditions) else COLOR_LIGHTNING if lightningConditions else (COLOR_LIFR_FADE if FADE_INSTEAD_OF_BLINK else COLOR_CLEAR) if windy else COLOR_CLEAR
				colorName = "Magenta"
			elif conditions["flightCategory"] == None:
				color = COLOR_UNK if not (windy or lightningConditions) else COLOR_LIGHTNING if lightningConditions else (COLOR_UNK_FADE if FADE_INSTEAD_OF_BLINK else COLOR_CLEAR) if windy else COLOR_CLEAR
				colorName = "Clear"
			else:
				color = COLOR_CLEAR
		if (windy or lightningConditions):
			print("Setting LED " + str(i) + " for " + airport + " to " + ("lightning " if lightningConditions else "") + ("windy " if windy else "") + (fltCat if conditions != None else "None") + " " + colorName)

		i += 1
		
    # Switching between animation cycles
	time.sleep(BLINK_SPEED)
	windCycle = False if windCycle else True
	looplimit -= 1

Loop 600 of 600
Setting LED 225 for KBGR to lightning LIFR Magenta
Loop 599 of 600
Setting LED 3 for KHRL to windy VFR Green
Setting LED 12 for KGCK to windy VFR Green
Setting LED 13 for KWWR to windy VFR Green
Setting LED 14 for KCSM to windy VFR Green
Setting LED 21 for KLBB to windy VFR Green
Setting LED 22 for KCDS to windy VFR Green
Setting LED 23 for KAMA to windy VFR Green
Setting LED 24 for KCAO to windy VFR Green
Setting LED 25 for KCVS to windy VFR Green
Setting LED 26 for KROW to windy VFR Green
Setting LED 39 for KTUS to windy VFR Green
Setting LED 46 for KSAN to windy VFR Green
Setting LED 80 for KSEA to windy VFR Green
Setting LED 89 for KCTB to windy VFR Green
Setting LED 90 for KGTF to windy VFR Green
Setting LED 91 for KBTM to windy VFR Green
Setting LED 93 for KPNA to windy VFR Green
Setting LED 94 for KRKS to windy VFR Green
Setting LED 96 for KCYS to windy VFR Green
Setting LED 97 for KCPR to windy VFR Green
Setting LED 98 for KGCC to windy VFR Green
Setting LED 101

KeyboardInterrupt: 